### 1. Load the cleaned data

In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx


PROCESSED_DATA_DIR = Path("../data/processed")

clean_edges = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_story_keyword_edges.csv"
)

print(clean_edges.shape)
display(clean_edges.head())

(20720, 3)


,story_id,keyword_id,keyword
0,1319,929,Buttern
1,1453,30,on cattle
2,15342,206,Etymologie
3,5811,5813,Herrenhaus
4,3616,29,witches


### 2. Group stories by keyword

In [2]:
keyword_to_stories = (
    clean_edges
    .groupby("keyword_id")["story_id"]
    .apply(list)
)

print("Keywords:", len(keyword_to_stories))

Keywords: 2021


### 3. Count shared keywords between stories

In [3]:
from itertools import combinations
from collections import Counter


story_pair_weights = Counter()

for stories in keyword_to_stories:
    unique_stories = sorted(set(stories))

    for story_a, story_b in combinations(unique_stories, 2):
        story_pair_weights[(story_a, story_b)] += 1

print("Story pairs sharing at least one keyword:", len(story_pair_weights))

Story pairs sharing at least one keyword: 1047739


### 4. Build the NetworkX graph

In [4]:
G = nx.Graph()

# Add all stories with keyword information.
G.add_nodes_from(clean_edges["story_id"].unique())

# Add weighted edges.
for (story_a, story_b), shared_keywords in story_pair_weights.items():
    G.add_edge(
        story_a,
        story_b,
        weight=shared_keywords
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 4577
Edges: 1047739


### 5. Remove weak connections

In [5]:
MIN_SHARED_KEYWORDS = 3

G_filtered = nx.Graph()

G_filtered.add_nodes_from(G.nodes())

for story_a, story_b, data in G.edges(data=True):
    if data["weight"] >= MIN_SHARED_KEYWORDS:
        G_filtered.add_edge(
            story_a,
            story_b,
            weight=data["weight"]
        )

# Remove stories that have no remaining connections.
G_filtered.remove_nodes_from(
    list(nx.isolates(G_filtered))
)

print("Filtered nodes:", G_filtered.number_of_nodes())
print("Filtered edges:", G_filtered.number_of_edges())

Filtered nodes: 4148
Filtered edges: 114644


### 6. Print basic graph statistics

In [6]:
print("Number of nodes:", G_filtered.number_of_nodes())
print("Number of edges:", G_filtered.number_of_edges())

print(
    "Number of connected components:",
    nx.number_connected_components(G_filtered)
)

largest_component = max(
    nx.connected_components(G_filtered),
    key=len
)

print("Largest component size:", len(largest_component))

average_degree = (
    sum(dict(G_filtered.degree()).values())
    / G_filtered.number_of_nodes()
)

print("Average degree:", round(average_degree, 2))

Number of nodes: 4148
Number of edges: 114644
Number of connected components: 122
Largest component size: 1266
Average degree: 55.28


### 7. Keep the largest connected component

In [7]:
largest_component_nodes = max(
    nx.connected_components(G_filtered),
    key=len
)

G_main = G_filtered.subgraph(
    largest_component_nodes
).copy()

print("Main graph nodes:", G_main.number_of_nodes())
print("Main graph edges:", G_main.number_of_edges())

Main graph nodes: 1266
Main graph edges: 31491


### 8. Save the graph

In [8]:
GRAPH_PATH = (
    PROCESSED_DATA_DIR
    / "story_similarity_graph.graphml"
)

nx.write_graphml(
    G_main,
    GRAPH_PATH
)

print("Saved graph to:")
print(GRAPH_PATH)

Saved graph to:
..\data\processed\story_similarity_graph.graphml


## Graph construction conclusion

A story-similarity graph was created from the cleaned keyword data.

Each node represents one story. Two stories are connected when they share
keywords, and the edge weight equals the number of shared keywords.

Edges based on only one shared keyword were removed to reduce weak connections.
The largest connected component was saved for community detection.